In [1]:
import Tensor as t
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml

We will stick to the row major order to align with numpy. This means our "dense" layers will be

$$\bold{Y} = \bold{X}\bold{W} + \bold{b}$$

Where $\bold{X}$ is the row vector in question. Might implement a technique called batching in the future.

In [2]:
# Fetch the MNIST dataset (this might take a minute to download)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Split into features (images) and labels
X, y = mnist["data"], mnist["target"]

print(f"Dataset shape: {X.shape}")

Dataset shape: (70000, 784)


In [3]:
def encode(val):
    z = np.zeros(10)
    z[int(val)] += 1
    return z

y_cleaned = np.array([np.array([encode(k)]) for k in y])

x_cleaned = X/255
x_cleaned = np.array([np.array([k]) for k in x_cleaned])

In [4]:
print(x_cleaned.shape)

(70000, 1, 784)


In [5]:
print(y_cleaned.shape)

(70000, 1, 10)


In [ ]:
#current architecture: Dense(784 15) sAct softmax Dense(15 10) sAct softmax (done!)
#paramaters
w_1 = np.random.random_sample(size=(784, 15))
b_1 = np.random.random_sample(size=(1,15))

w_2 = np.random.random_sample(size=(15,10))
b_2 = np.random.random_sample(size=(1,10))

def pipeline(input):
    L_1 = (t.TensorNode(input,is_param=False).copy()) @ t.TensorNode(w_1) + t.TensorNode(b_1)
    L_1 = L_1.sAct().sigmoid()

    L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)

    L_2 = L_2.sAct().sigmoid()
    return L_2

In [ ]:
model = pipeline(x_cleaned[0])
model.compile() 
#use model to actually get predictions and vector ouputs!

[[0.11895899 0.10362753 0.10764323 0.11579551 0.10565337 0.08585548
  0.10638829 0.07685516 0.10121536 0.07800709]]
0.9999999999952005


In [8]:
loss = (pipeline(x_cleaned[0]) - t.TensorNode(y_cleaned[0],is_param=False)).norm_squared()
print(loss.data)

0.9302632437437464


In [9]:
loss.compile()
losses = [loss]
for i in range(1,50_000):
    losses.append((pipeline(x_cleaned[i]) - t.TensorNode(y_cleaned[i],is_param=False)).norm_squared())
    losses[-1].compile()


In [10]:
#now losses will give us ultimate control over how we wanna batch up training.
v = 0
for lossy in losses:
    
    v += lossy.data
print(v)

45088.21712816763


In [68]:
prevLoss = 4278.051787650891
for k in range(3):
    currLoss = 0
    for lossy in losses:
        lossy.train()
        lossy.update(0.125)
        loss.compute(loss.topo_sort[-1][0].data)



In [69]:
#now losses will give us ultimate control over how we wanna batch up training.
#this is the squared error loss.
v = 0
for lossy in losses:
    v += lossy.data
print(v)

2890.658309452404


WE DID IT. Below is the accuracy:

In [85]:
correct = 0
for j in range(50_000):
    model.compute(x_cleaned[j])
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/50_000))

print("test dataset:")
correct = 0
for j in range(50_001,70_000):
    model.compute(x_cleaned[j])
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/20_000))


Raw correct: 48233
accruacy 0.96466
test dataset:
Raw correct: 18975
accruacy 0.94875


Check out data.npz to import paramaters

In [99]:
ohio = np.sum(x_cleaned,axis=1)
print(ohio.shape)

z = pipeline(ohio)
print(z.data.shape)

model.compute(x_cleaned[0])
print(model.data)


(70000, 784)
(70000, 10)
[[5.86878201e-05 3.74427247e-06 8.45229891e-04 1.14523442e-02
  2.09270227e-06 9.87502060e-01 5.28566438e-08 1.32436383e-04
  1.11784666e-06 2.23395239e-06]]


[[2.54621861e-07 1.64666937e-07 9.27453243e-06 4.30300826e-07
  6.07674160e-08 3.48028300e-06 2.75468412e-08 5.78835710e-07
  1.00806914e-08 4.76950339e-09]]

In [96]:
test = ohio @ w_1 + b_1
test_2 = x_cleaned[5] @ w_1 + b_1
print(test[5])
print(test_2)

[-39.38191836 -27.42794958  11.39429914   9.51483085 -15.20732144
 -13.94561886   5.91993219 -21.40566942 -36.17214612 -25.67809893
  22.61197527 -11.06204503  -7.57028274  45.06873197 -29.45799848]
[[-39.38191836 -27.42794958  11.39429914   9.51483085 -15.20732144
  -13.94561886   5.91993219 -21.40566942 -36.17214612 -25.67809893
   22.61197527 -11.06204503  -7.57028274  45.06873197 -29.45799848]]
